# 6. Deployment
We train the forest + L1 method chosen in Modeling using all 6,469 training
purchases with a date, then create the Kaggle CSV. We keep the same settings
used in the development comparison.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from preparation import fill_data, scale_data, encode_data

## Load the clean data
We read the training and test tables saved in Data Preparation. Training
includes the development, gap and holdout periods. The 10 duplicates and
55 undated rows remain excluded. Test keeps all IDs in their original order.

In [2]:
train = pd.read_csv("../data/clean/train.csv", parse_dates=["purchase_date"])
test = pd.read_csv("../data/clean/test.csv", parse_dates=["purchase_date"])
target = "repeat_purchase_90d"
test_ids = test["ID"].copy()

train = train.sort_values(["purchase_date", "ID"])
print("Eligible training purchases:", len(train))
print("Test purchases:", len(test))
print("Days between last training and first test purchase:",
      (test["purchase_date"].min() - train["purchase_date"].max()).days)

Eligible training purchases: 6469
Test purchases: 986
Days between last training and first test purchase: 98


There are 6,469 training purchases. Test starts 98 days after the last one,
so all training labels would already be known before the first test purchase.
We can now also use the 147 purchases previously left in the 90-day gap.

## Create the same calendar variables
We create year, month and weekday as in the previous notebooks.

In [3]:
for df in [train, test]:
    df["purchase_year"] = df["purchase_date"].dt.year
    df["purchase_month"] = df["purchase_date"].dt.month.astype(str)
    df["purchase_weekday"] = df["purchase_date"].dt.dayofweek.astype(str)

train = train.set_index("ID")
test = test.set_index("ID")
y_train = train[target]
X_train = train.drop(columns=["customer_id", "purchase_date", target])
X_test = test.drop(columns=["customer_id", "purchase_date"])

## Prepare the full training data
We fit missing-value treatment, scaling and encoding again on the full training
set. Test uses those same parameters. More categories may now reach the
minimum frequency, so the number of encoded columns can change.

In [4]:
X_train, X_test, num, cat = fill_data(X_train, X_test)
X_train, X_test = scale_data(X_train, X_test, num)
X_train, X_test = encode_data(X_train, X_test, cat)
print("Full training inputs:", X_train.shape)
print("Test inputs:", X_test.shape)

Full training inputs: (6469, 302)
Test inputs: (986, 302)


## Select variables again with L1
We use the same selector with `C=0.1`, now fitted on all training purchases.
We choose the variables again because the available encoded columns can differ
from those in development.

In [5]:
select = LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                             max_iter=2000, random_state=42)
select.fit(X_train, y_train)
keep = np.abs(select.coef_[0]) > 0.000001
cols = X_train.columns[keep].tolist()
print("Inputs used in final fitting:", len(cols))

Inputs used in final fitting: 58


In [6]:
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                random_state=42, n_jobs=2)
model.fit(X_train[cols], y_train)
pred = model.predict(X_test[cols])

## Create and check the submission
We keep the original purchase IDs and add one 0/1 prediction for each row.
The sample submission shows the required format; its labels are examples.

In [7]:
sub = pd.DataFrame({"ID": test.index, target: pred.astype(int)})
print("Submission shape:", sub.shape)
print("Repeated IDs:", sub["ID"].duplicated().sum())
print("Predicted class counts:")
print(sub[target].value_counts().sort_index())
sub.head()

Submission shape: (986, 2)
Repeated IDs: 0
Predicted class counts:
repeat_purchase_90d
0    341
1    645
Name: count, dtype: int64


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1


In [8]:
sample = pd.read_csv("../data/raw/sample_submission.csv")
assert sub["ID"].tolist() == test_ids.tolist()
assert sub.columns.tolist() == sample.columns.tolist()
assert set(sub[target].unique()).issubset({0, 1})

## Save version 01
We save the CSV without an extra index column. We keep this version number
when reproducing the same solution and change it for a new solution.

In [9]:
os.makedirs("../submissions", exist_ok=True)
version = "01"
path = "../submissions/ML_Group_06_Version" + version + ".csv"
sub.to_csv(path, index=False)
pd.Series(cols, name="feature").to_csv("../results/final_features.csv", index=False)
print("Saved:", path)

Saved: ../submissions/ML_Group_06_Version01.csv


## Submission status
The full training data produce 302 prepared variables, of which L1 keeps 58.
The CSV has 986 rows: 645 predictions of class 1 and 341 of class 0.
These counts do not tell us how accurate the predictions are.

The file is saved locally. Kaggle upload and final submission selection are
still pending. Use team `ML_Group_06` and keep the versioned filename.
The public leaderboard uses 70% of test rows; the other 30% determine the
private leaderboard. The submitted file must match the solution in the notebook.

The two-page homework report is also prepared. The final project still needs
at least five taught algorithms, optimisation and an additional investigation.